# Diagnosing Models with TensorBoard

Train a small classifier on the same credit-card transaction dataset used in [deep_learning.ipynb](deep_learning.ipynb), log it with Keras's `TensorBoard` callback, then use TensorBoard to (1) diagnose *that* run -- is it over/underfitting, are any layers dead or exploding -- and (2) sweep several architectures through the **HParams** dashboard to decide which model *structure* actually generalizes best, rather than guessing.

Dataset: ~285,000 anonymized European card transactions (`Time`, `Amount`, PCA features `V1`-`V28`, label `Class`: 1 = fraud), hosted by TensorFlow at `storage.googleapis.com`.

In [1]:
import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)

I0000 00:00:1788795716.023327  241046 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788795716.023759  241046 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788795716.061799  241046 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788795716.994138  241046 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788795716.994417  241046 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Load the transaction data

In [2]:
csv_path = tf.keras.utils.get_file(
    "creditcard.csv",
    "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv",
)
df = pd.read_csv(csv_path)

feature_cols = [c for c in df.columns if c != "Class"]
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Chronological train/val/test split and scaling

Transactions are already ordered by `Time`, so we split chronologically (train on the past, evaluate on the future) rather than shuffling at random.

In [3]:
n = len(df)
train_end, val_end = int(n * 0.7), int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

scaler = StandardScaler().fit(train_df[feature_cols])

X_train = scaler.transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])

y_train = train_df["Class"].to_numpy()
y_val = val_df["Class"].to_numpy()
y_test = test_df["Class"].to_numpy()

print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")
print(f"fraud rate (train) = {y_train.mean():.4%}")

train=199,364  val=42,721  test=42,722
fraud rate (train) = 0.1926%


## Model

In [4]:
cb =  tf.keras.callbacks.ReduceLROnPlateau()

In [5]:
def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,),name="input"),
        tf.keras.layers.Dense(64, activation="relu",name="dense_1"),
        # tf.keras.layers.Dropout(0.3,name="Dropout"),
        tf.keras.layers.Dense(16, activation="relu",name="Dense_2"),
        tf.keras.layers.Dense(1, activation="sigmoid",name="output"),
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="acurracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
        
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

E0000 00:00:1788795719.455277  241046 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_2 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,041 (11.88 KB)

 Trainable params: 3,041 (11.88 KB)

 Non-trainable params: 0 (0.00 B)

## TensorBoard callback

Each run gets its own log directory under `logs/fit/`, named `<run_name>_<timestamp>` -- the timestamp keeps runs unique, and `run_name` (architecture + batch size + optimizer) is what actually lets you tell runs apart in the TensorBoard sidebar instead of a bare timestamp. `histogram_freq=1` also logs per-epoch weight/activation histograms. The model graph itself needs no extra setup: `write_graph=True` (the default) already logs it to the **Graphs** tab.

In [6]:
run_name = "test 1 model"
log_dir = f"logs/fit/{run_name}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
)

# class weights so the ~0.17% fraud rate doesn't get ignored by the optimizer
neg, pos = np.bincount(y_train)
class_weight = {0: (1 / neg) * (len(y_train) / 2.0), 1: (1 / pos) * (len(y_train) / 2.0)}

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=2048,
    class_weight=class_weight,
    callbacks=[tensorboard_callback,tf.keras.callbacks.ReduceLROnPlateau()],
    verbose=2,
)

Epoch 1/15


98/98 - 1s - 13ms/step - acurracy: 0.8933 - loss: 0.4793 - precision: 0.0134 - recall: 0.7500 - val_acurracy: 0.9846 - val_loss: 0.2441 - val_precision: 0.0710 - val_recall: 0.8929 - learning_rate: 0.0010


Epoch 2/15


98/98 - 0s - 3ms/step - acurracy: 0.9824 - loss: 0.2050 - precision: 0.0900 - recall: 0.8906 - val_acurracy: 0.9838 - val_loss: 0.1358 - val_precision: 0.0678 - val_recall: 0.8929 - learning_rate: 0.0010


Epoch 3/15


...

98/98 - 0s - 3ms/step - acurracy: 0.9857 - loss: 0.0620 - precision: 0.1156 - recall: 0.9688 - val_acurracy: 0.9903 - val_loss: 0.0339 - val_precision: 0.1109 - val_recall: 0.9107 - learning_rate: 0.0010


Epoch 14/15


98/98 - 0s - 3ms/step - acurracy: 0.9862 - loss: 0.0577 - precision: 0.1201 - recall: 0.9740 - val_acurracy: 0.9905 - val_loss: 0.0329 - val_precision: 0.1131 - val_recall: 0.9107 - learning_rate: 0.0010


Epoch 15/15


98/98 - 0s - 3ms/step - acurracy: 0.9868 - loss: 0.0535 - precision: 0.1253 - recall: 0.9740 - val_acurracy: 0.9910 - val_loss: 0.0304 - val_precision: 0.1183 - val_recall: 0.9107 - learning_rate: 0.0010


## Launch TensorBoard

Run this in a notebook cell (Jupyter) to view the logged run inline. From a terminal instead, run `tensorboard --logdir logs/fit`.

In [7]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 72523), started 1:58:21 ago. (Use '!kill 72523' to kill it.)

## Test set evaluation

In [8]:
results = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
results

{'acurracy': 0.9938673377037048,
 'loss': 0.02025342732667923,
 'precision': 0.14765100181102753,
 'recall': 0.8461538553237915}

## Diagnosing this run in TensorBoard

**Scalars tab -- reading the fit.** Compare `epoch_loss` against `val_epoch_loss` epoch over epoch:

- **overfitting**: train loss keeps falling while val loss flattens out or starts climbing back up -- a widening gap between the two curves.
- **underfitting**: both curves flatten early, at a high loss -- the model (or learning rate, or epoch budget) is too small for the problem.
- **good fit**: both curves fall together and stay close.
- **noisy validation curve**: use the "Smoothing" slider before reading anything into it -- a validation set of a few thousand examples is naturally noisier than a training set of hundreds of thousands.


## Pathology gallery: seeing vanishing/exploding gradients and dead ReLUs

The healthy run above is a useful baseline, but it's much easier to recognize a pathological histogram once you've actually seen one next to a healthy one. Below we deliberately build three broken models -- one with **vanishing gradients**, one with **exploding gradients**, one with **dead ReLUs** -- on a small synthetic dataset (so the pathology isn't tangled up with the fraud data's class imbalance), and log every one with the same `TensorBoard` callback used throughout this notebook, each to its own run folder under `logs/fit/pathology_*` in the TensorBoard instance already running above.

In [9]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_synth, y_synth = make_classification(
    n_samples=4000, n_features=20, n_informative=15, n_redundant=2,
    class_sep=1.0, random_state=42,
)
X_synth = StandardScaler().fit_transform(X_synth).astype("float32")
y_synth = y_synth.astype("float32")

# held-out split so the fit-quality gallery below (which needs a val curve) can reuse it too
X_synth_train, X_synth_val, y_synth_train, y_synth_val = train_test_split(
    X_synth, y_synth, test_size=0.25, random_state=42, stratify=y_synth
)


def deep_model(activation, depth, width=32, dropout=0.0, kernel_initializer="glorot_uniform",
               bias_initializer="zeros", optimizer=None):
    layers = [tf.keras.layers.Input(shape=(X_synth.shape[1],), name="input")]
    for i in range(depth):
        layers.append(tf.keras.layers.Dense(
            width, activation=activation, name=f"dense_{i + 1}",
            kernel_initializer=kernel_initializer, bias_initializer=bias_initializer,
        ))
        if dropout:
            layers.append(tf.keras.layers.Dropout(dropout, name=f"dropout_{i + 1}"))
    layers.append(tf.keras.layers.Dense(1, activation="sigmoid", name="output"))
    model = tf.keras.Sequential(layers)
    model.compile(
        optimizer=optimizer or tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [10]:
pathology_root = f"logs/fit/pathology_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

pathology_configs = {
    "control": dict(activation="relu", depth=6, width=32),
    "vanishing": dict(activation="sigmoid", depth=15, width=32),
    "exploding": dict(
        activation="relu", depth=10, width=32,
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=1.5),
        optimizer=tf.keras.optimizers.SGD(learning_rate=1.0),
    ),
    "dead_relu": dict(activation="relu", depth=6, width=32,
                       bias_initializer=tf.keras.initializers.Constant(-10.0)),
}

for name, cfg in pathology_configs.items():
    model = deep_model(**cfg)
    history = model.fit(
        X_synth, y_synth,
        epochs=15,
        batch_size=128,
        verbose=0,
        callbacks=[tf.keras.callbacks.TensorBoard(log_dir=f"{pathology_root}/{name}", histogram_freq=1)],
    )
    print(f"{name:>10}: final loss = {history.history['loss'][-1]:.4g}")

   control: final loss = 0.06107


 vanishing: final loss = 0.6943


 exploding: final loss = nan


 dead_relu: final loss = 0.6932


### Reading it back in TensorBoard

All four runs above logged to `logs/fit/pathology_<timestamp>/<name>/`, in the same TensorBoard instance already running (`--logdir logs/fit`) -- refresh it and filter the run list by `pathology` to isolate them.

- **control** -- Scalars: loss falls smoothly. Histograms: `dense_1/kernel` through `dense_6/kernel` all visibly widen/shift epoch over epoch -- every layer is learning.
- **vanishing** (`sigmoid`, 15 layers) -- Scalars: loss sits flat around `ln(2) ~= 0.69` (the loss of a coin flip) for the whole run -- the network never learns anything. Histograms: `dense_15/kernel` (closest to the output) moves a little; `dense_1` through roughly `dense_9` are visibly frozen at their initial values the entire run. This is what an uninterrupted stack of saturating activations (`sigmoid`/`tanh`) does: each layer's derivative is at most 0.25 (sigmoid) or 1.0 (tanh), and the chain rule multiplies ~15 of those together on the way back to the early layers.
- **exploding** (large init + `SGD(lr=1.0)`) -- Scalars: loss shoots up to an enormous value, or reads `NaN` outright, within the first epoch or two -- the weights have overflowed to `inf`/`NaN`, and everything computed from them afterward is meaningless, not converged. Histograms: watch epoch 0 -> 1 specifically -- a layer's distribution fans out to a huge range, and the epoch after that there's nothing finite left to render.
- **dead ReLU** (`bias_initializer=-10`) -- Scalars: loss sits flat at `ln(2)` from epoch 1 onward, identical to vanishing but for a different reason. Histograms: every weight distribution is frozen at its initial shape from epoch 1 onward -- the large negative bias pushes every pre-activation below 0, `relu` zeroes it, and a zeroed activation has zero gradient, so the optimizer never has anything to update.

Common fixes, matched to the tell: **vanishing** -> non-saturating activations (`relu`/`gelu`), depth-aware init (`he_normal` for relu), residual connections, or simply fewer layers; **exploding** -> lower learning rate, gradient clipping (`clipnorm=`), or a smaller/careful init; **dead ReLUs** -> lower learning rate, `he_normal` init, a small positive bias init, or a leaky variant (`LeakyReLU`).

## Fit-quality gallery: seeing overfitting and underfitting

Same idea as the gradient pathologies above, applied to the fit-quality patterns from the Scalars section earlier: instead of just describing "train and val curves diverge," here are three real models on the same synthetic dataset (with its `X_synth_train`/`X_synth_val` split from above) tuned to visibly **overfit**, visibly **underfit**, and fit well -- each logged with the standard `TensorBoard` callback so the three curve shapes can be compared directly in its Scalars tab.

In [11]:
fit_configs = {
    "good_fit": dict(
        cfg=dict(activation="relu", depth=2, width=16, dropout=0.2),
        train_x=X_synth_train, train_y=y_synth_train, epochs=30, batch_size=64,
    ),
    "overfitting": dict(
        # a high-capacity model with almost no training data to generalize from
        cfg=dict(activation="relu", depth=4, width=256),
        train_x=X_synth_train[:100], train_y=y_synth_train[:100], epochs=60, batch_size=16,
    ),
    "underfitting": dict(
        # a 2-unit bottleneck plus a learning rate too small to escape it in time
        cfg=dict(activation="relu", depth=1, width=2, optimizer=tf.keras.optimizers.Adam(1e-4)),
        train_x=X_synth_train, train_y=y_synth_train, epochs=30, batch_size=64,
    ),
}

fit_root = f"logs/fit/fit_gallery_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
for name, spec in fit_configs.items():
    model = deep_model(**spec["cfg"])
    history = model.fit(
        spec["train_x"], spec["train_y"],
        validation_data=(X_synth_val, y_synth_val),
        epochs=spec["epochs"],
        batch_size=spec["batch_size"],
        verbose=0,
        callbacks=[tf.keras.callbacks.TensorBoard(log_dir=f"{fit_root}/{name}", histogram_freq=1)],
    )
    print(f"{name:>12}: final train loss = {history.history['loss'][-1]:.3f}   "
          f"final val loss = {history.history['val_loss'][-1]:.3f}")

    good_fit: final train loss = 0.219   final val loss = 0.231


 overfitting: final train loss = 0.000   final val loss = 0.808


underfitting: final train loss = 0.659   final val loss = 0.683


### Reading it back in TensorBoard

All three runs logged to `logs/fit/fit_gallery_<timestamp>/<name>/`, same TensorBoard instance (`--logdir logs/fit`) -- filter the run list by `fit_gallery`.

- **good_fit** -- Scalars: `epoch_loss` and `val_epoch_loss` fall together and end close to each other (a small dropout-shaped gap, not a widening one) -- the "both curves fall together and stay close" pattern from the diagnosis section above.
- **overfitting** -- Scalars: `epoch_loss` keeps falling all the way toward 0 while `val_epoch_loss` bottoms out early and climbs back up for the rest of training -- hover the val curve to find exactly which epoch it turns; an early-stopping callback watching `val_loss` would have frozen the model right there. Only ~100 training examples feeding a 4-layer, 256-wide network is the cause here -- capacity far exceeding the data available to constrain it.
- **underfitting** -- Scalars: both curves flatten early, close together, at a high loss (near `ln 2` again) -- "close together" is what separates this from overfitting; the model isn't memorizing anything, it simply isn't learning much of anything, train or val. Fixes to reach for, roughly in order of trying first: raise the learning rate, add capacity (more units/layers), train for longer -- and only once those are exhausted, suspect the data itself doesn't contain enough signal.